# Recitation 0.2 : Object-Oriented Programming (OOP) for Deep Learning

TA: Hrishikesh (Chris, if too complicated) Bhagwat (hbhagwat@andrew.cmu.edu)

Welcome back!


**Prerequisite:** Part 1 and the ability to run Jupyter notebooks.

## 1. Why OOP?



In [57]:
# Procedural approach
p1_x, p1_y = 1.0, 2.0
p2_x, p2_y = 3.0, 4.0

def distance_from_origin(x, y):
    return (x**2 + y**2) ** 0.5

def add_points(x1, y1, x2, y2):
    return x1 + x2, y1 + y2

print(distance_from_origin(p1_x, p1_y))
print(add_points(p1_x, p1_y, p2_x, p2_y))

2.23606797749979
(4.0, 6.0)


Things get complicated when we move to 3D! So we make use of classes and objects!

## 2. Classes and Objects

- A **class** is a **mould**.
- An **object** (or **instance**) is a **casting** built from that mould.

### In Python, everything is an object!

Even the "primitive" values you used in Part 1 are instances of classes. This isn't trivia, it's the reason the same syntax works on ints, strings, NumPy arrays.

In [58]:
a = "1"
print(type(a))

a = 1
print(type(a))

<class 'str'>
<class 'int'>


### Let us do it the OOP way!

In [59]:
class Point:
    def __init__(self, x: float, y: float) -> None:  # constructor
        self.x = x   # instance attribute
        self.y = y   # instance attribute

    def distance_from_origin(self) -> float:          # instance method
        return (self.x**2 + self.y**2) ** 0.5

    def describe(self) -> None:
        print(f"Point({self.x}, {self.y}) — distance from origin: {self.distance_from_origin():.2f}")


p1 = Point(1.0, 2.0)   # instantiation
p2 = Point(3.0, 4.0)

print(p1)

print(type(p1))         # <class '__main__.Point'>
print(p1.x, p1.y)      # access attributes with "."
p1.describe()           # call method with "."
p2.describe()

<class '__main__.Point'>
1.0 2.0
Point(1.0, 2.0) — distance from origin: 2.24
Point(3.0, 4.0) — distance from origin: 5.00


### 2.1 The constructor and `self`

- `__init__` is a **dunder method**. Python calls it automatically when you create an object.
- `self` refers to **this specific instance**. It's how the object keeps track of its own data.

```python
p1 = Point(1.0, 2.0)
#            ^    ^
#            x    y
# Python passes self=p1 automatically behind the scenes
```

### 2.2 Type hints, docstrings, default values

Type hints and docstrings aren't enforced at runtime (just like in Part 1), but every piece of homework starter code uses them. Default argument values let a caller omit arguments they don't need.

In [60]:
class Point:
    def __init__(self, x: float, y: float, z: float = 0.0) -> None:
        """
        Args:
            x, y: Coordinates in 2D space.
            z:    Optional third dimension (default: 0.0).
        """
        self.x = x
        self.y = y
        self.z = z

    def distance_from_origin(self) -> float:
        """Euclidean distance from the origin."""
        return (self.x**2 + self.y**2 + self.z**2) ** 0.5

    def describe(self) -> None:
        """Print coordinates and distance."""
        print(f"Point({self.x}, {self.y}, {self.z}) — dist: {self.distance_from_origin():.2f}")


p1 = Point(1.0, 2.0)          # z defaults to 0.0
p2 = Point(1.0, 2.0, 3.0)

p1.describe()
p2.describe()

Point(1.0, 2.0, 0.0) — dist: 2.24
Point(1.0, 2.0, 3.0) — dist: 3.74


## 3. Core Concepts: The Four Pillars

OOP rests on four ideas: **Encapsulation**, **Abstraction**, **Inheritance**, and **Polymorphism**. We'll build each one up with a running `BankAccount` example, then apply them.

### 3.1 Encapsulation

Two ideas:
1. **Bundle** data and methods inside a class.
2. **Control access**. Hide internal state, expose a clean interface.

Python uses naming conventions rather than hard access modifiers like C++:

| Convention | Meaning |
|---|---|
| `attribute` | Public — accessible anywhere |
| `_attribute` | Protected — internal use (by convention) |
| `__attribute` | Private — name-mangled, actively discouraged from outside |

In [61]:
class BankAccount:
    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner    = owner           # public
        self._balance = balance         # protected — managed through methods
        self.__pin    = 1234            # private — should never be accessed directly

    def get_balance(self) -> float:     # getter
        return self._balance

    def set_balance(self, amount: float) -> None:   # setter with validation
        if amount < 0:
            raise ValueError("Balance cannot be negative.")
        self._balance = amount

    def describe(self) -> None:
        print(f"{self.owner}'s account — balance: ${self._balance:.2f}")


acc = BankAccount("Alice", 500.0)

print(acc.owner)           # public — fine
print(acc.get_balance())   # controlled access via getter

acc.set_balance(750.0)     # validated update
acc.describe()

acc.__pin              # AttributeError — private, can't touch it

Alice
500.0
Alice's account — balance: $750.00


AttributeError: 'BankAccount' object has no attribute '__pin'

The setter lets you **validate before changing**. Direct access (`acc._balance = -999`) skips all that, which is exactly why the underscore convention exists: it signals "don't reach in here directly."

### 3.2 Abstraction

**Abstraction** means hiding complexity behind a clean interface. The user calls simple methods and doesn't need to know what happens inside.

Think of an ATM: you press "Withdraw", you get cash. You never see the ledger update or the validation logic.

Internal helpers are prefixed with `_` to signal they're not part of the public interface.

In [62]:
class BankAccount:
    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner         = owner
        self._balance      = balance
        self._transactions = []

    # ---- Internal helpers — not for users ----
    def _validate_amount(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Amount must be positive.")

    def _record(self, tx_type: str, amount: float) -> None:
        self._balance += amount
        self._transactions.append({"type": tx_type, "amount": amount, "balance": self._balance})

    # ---- Public interface ----
    def deposit(self, amount: float) -> None:
        """Deposit money into the account."""
        self._validate_amount(amount)
        self._record("DEPOSIT", amount)

    def withdraw(self, amount: float) -> None:
        """Withdraw money from the account."""
        self._validate_amount(amount)
        if amount > self._balance:
            print("Insufficient funds.")
            return
        self._record("WITHDRAW", -amount)

    def statement(self) -> None:
        """Print full transaction history."""
        print(f"Statement — {self.owner}")
        print("-" * 42)
        for i, tx in enumerate(self._transactions, 1):
            print(f"{i}. {tx['type']:8} | {tx['amount']:>8.2f} | Balance: {tx['balance']:>8.2f}")
        print("-" * 42)
        print(f"Current balance: ${self._balance:.2f}")


acc = BankAccount("Alice", balance=0)
acc.deposit(1000)
acc.deposit(500)
acc.withdraw(200)
acc.withdraw(9999)   # should fail gracefully

print()
acc.statement()

Insufficient funds.

Statement — Alice
------------------------------------------
1. DEPOSIT  |  1000.00 | Balance:  1000.00
2. DEPOSIT  |   500.00 | Balance:  1500.00
3. WITHDRAW |  -200.00 | Balance:  1300.00
------------------------------------------
Current balance: $1300.00


The user only calls `deposit()`, `withdraw()`, `statement()`. The internal `_validate_amount` and `_record` helpers are hidden. That's abstraction.

You can also enforce abstraction formally with `@abstractmethod`. Read more [here](https://www.geeksforgeeks.org/python/abstract-classes-in-python/).

### 3.3 Inheritance

**Inheritance** lets a child class **reuse and extend** a parent class.

- **Parent class** — general blueprint
- **Child class** — more specific; inherits the parent's attributes and methods via `super()`
- Children can **override** parent methods to change behavior.

Keep an eye on `super().__init__(...)` below. It's the single most important line to understand before you touch PyTorch.

In [63]:
# Parent
class BankAccount:
    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner    = owner
        self._balance = balance

    def deposit(self, amount: float) -> None:
        self._balance += amount

    def get_balance(self) -> float:
        return self._balance

    def describe(self) -> None:
        print(f"[Account] {self.owner} — ${self._balance:.2f}")


# Child — adds interest
class SavingsAccount(BankAccount):
    def __init__(self, owner: str, balance: float, interest_rate: float) -> None:
        super().__init__(owner, balance)     # inherit parent attributes
        self.interest_rate = interest_rate   # new attribute

    def apply_interest(self) -> None:        # new method
        self._balance *= (1 + self.interest_rate)

    def describe(self) -> None:              # override
        print(f"[Savings] {self.owner} — ${self._balance:.2f} @ {self.interest_rate:.0%} interest")


# Grandchild — adds overdraft protection
class PremiumAccount(SavingsAccount):
    def __init__(self, owner: str, balance: float, interest_rate: float, overdraft_limit: float) -> None:
        super().__init__(owner, balance, interest_rate)
        self.overdraft_limit = overdraft_limit

    def withdraw(self, amount: float) -> None:   # new method
        if amount > self._balance + self.overdraft_limit:
            print("Exceeds overdraft limit.")
            return
        self._balance -= amount

    def describe(self) -> None:
        print(f"[Premium] {self.owner} — ${self._balance:.2f}, overdraft up to ${self.overdraft_limit:.2f}")


a1 = BankAccount("Alice", 1000)
a2 = SavingsAccount("Bob", 2000, 0.05)
a3 = PremiumAccount("Carol", 500, 0.03, 200)

a1.describe()
a2.describe()
a3.describe()

a2.apply_interest()   # inherited _balance, new method
a2.describe()

a3.withdraw(650)      # uses overdraft
a3.describe()

[Account] Alice — $1000.00
[Savings] Bob — $2000.00 @ 5% interest
[Premium] Carol — $500.00, overdraft up to $200.00
[Savings] Bob — $2100.00 @ 5% interest
[Premium] Carol — $-150.00, overdraft up to $200.00


In [64]:
# isinstance() reflects the full inheritance chain
print(isinstance(a3, PremiumAccount))  # True
print(isinstance(a3, SavingsAccount))  # True
print(isinstance(a3, BankAccount))     # True

True
True
True


### 3.4 Polymorphism

**Polymorphism** ("many forms") means different classes share a method name, and you can call it uniformly without caring which specific type you're dealing with.

In [74]:
accounts = [
    BankAccount("Alice", 1000),
    SavingsAccount("Bob", 2000, 0.05),
    PremiumAccount("Carol", 500, 0.03, 200),
]

for acc in accounts:
    acc.describe()   # same call — different output per type

[Account] Alice — $1000.00
[Savings] Bob — $2000.00 @ 5% interest
[Premium] Carol — $500.00, overdraft up to $200.00


Python picks the right `describe()` based on the object's actual type at runtime, no `if/elif` needed. This is called **dynamic dispatch**, and it's what lets frameworks call `.forward()` on your model without knowing which model you wrote.

## 4. Dunder Methods and Operator Overloading

**Dunder methods** (double-underscore, also called *magic methods*) let you define how your objects behave with built-in Python operations: `print()`, `+`, `len()`, `==`, `<`, and so on.

You've already used one, `__init__`. There are many more.

We'll use a `Vector` class, a natural fit since vectors genuinely support addition, have a length, and can be compared by magnitude.

### 4.1 `__str__` — Readable `print()` output

Without it, `print(v)` gives something useless like `<__main__.Vector object at 0x7f...>`.

In [75]:
class Vector:
    def __init__(self, components: list) -> None:
        self.components = components

    def __str__(self) -> str:    # called by print() and str()
        return f"Vector({self.components})"


v = Vector([1, 2, 3])
print(v)       # Vector([1, 2, 3])
print(str(v))  # same

Vector([1, 2, 3])
Vector([1, 2, 3])


### 4.2 `__len__` — Support for `len()`

`len(v)` = number of dimensions in the vector. Completely natural.

In [76]:
class Vector:
    def __init__(self, components: list) -> None:
        self.components = components

    def __str__(self) -> str:
        return f"Vector({self.components})"

    def __len__(self) -> int:    # called by len()
        return len(self.components)


v1 = Vector([1, 2, 3])
v2 = Vector([4, 5, 6, 7])

print(len(v1))  # 3 — a 3D vector
print(len(v2))  # 4 — a 4D vector

3
4


### 4.3 `__add__` — Operator overloading with `+`

Vector addition is defined as element-wise sum, a perfect use of `__add__`.

In [77]:
class Vector:
    def __init__(self, components: list) -> None:
        self.components = components

    def __str__(self) -> str:
        return f"Vector({self.components})"

    def __len__(self) -> int:
        return len(self.components)

    def __add__(self, other: 'Vector') -> 'Vector':   # called by v1 + v2
        if len(self) != len(other):
            raise ValueError("Vectors must have the same number of dimensions.")
        result = [a + b for a, b in zip(self.components, other.components)]
        return Vector(result)


v1 = Vector([1, 2, 3])
v2 = Vector([4, 5, 6])

v3 = v1 + v2   # calls v1.__add__(v2)
print(v3)      # Vector([5, 7, 9])

Vector([5, 7, 9])


### 4.4 `__eq__` and `__lt__` — Comparison operators

Compare vectors by their **magnitude** (length in space). Once `__lt__` is defined, `sorted()` just works on your objects.

In [78]:
class Vector:
    def __init__(self, components: list) -> None:
        self.components = components

    def __str__(self) -> str:
        return f"Vector({self.components})"

    def __len__(self) -> int:
        return len(self.components)

    def __add__(self, other: 'Vector') -> 'Vector':
        if len(self) != len(other):
            raise ValueError("Vectors must have the same dimensions.")
        return Vector([a + b for a, b in zip(self.components, other.components)])

    def magnitude(self) -> float:
        return sum(x**2 for x in self.components) ** 0.5

    def __eq__(self, other: 'Vector') -> bool:   # ==
        return self.magnitude() == other.magnitude()

    def __lt__(self, other: 'Vector') -> bool:   # <
        return self.magnitude() < other.magnitude()


v1 = Vector([1, 0, 0])
v2 = Vector([0, 1, 0])
v3 = Vector([3, 4, 0])

print(v1 == v2)    # True — both have magnitude 1.0
print(v1 < v3)     # True — magnitude 1 < magnitude 5

# Once __lt__ is defined, sorted() works on your objects!
vectors = [v3, v1, v2]
for v in sorted(vectors):
    print(v, "— magnitude:", v.magnitude())

True
True
Vector([1, 0, 0]) — magnitude: 1.0
Vector([0, 1, 0]) — magnitude: 1.0
Vector([3, 4, 0]) — magnitude: 5.0


### Common dunder methods — quick reference

| Method | Triggered by | Typical use |
|---|---|---|
| `__init__` | `MyClass(...)` | Constructor |
| `__str__` | `print(obj)`, `str(obj)` | Human-readable string |
| `__len__` | `len(obj)` | Size or dimensionality |
| `__add__` | `obj1 + obj2` | Addition / merging |
| `__eq__` | `obj1 == obj2` | Equality check |
| `__lt__` | `obj1 < obj2` | Less-than; enables `sorted()` |
| `__getitem__` | `obj[i]` | Index access |
| `__call__` | `obj(...)` | Make an instance callable like a function |

Keep `__getitem__`, `__len__`, and `__call__` in mind for the next section, they're exactly the ones PyTorch relies on.

## 5. Class Attributes vs Instance Attributes

- **Instance attributes** belong to each object individually, defined via `self` in `__init__`.
- **Class attributes** are shared across all instances, defined at the class level.

In [79]:
class BankAccount:
    bank_name = "PyBank"   # class attribute — same for every account
    count     = 0          # class attribute — shared counter

    def __init__(self, owner: str, balance: float) -> None:
        self.owner    = owner     # instance attribute — unique per object
        self._balance = balance
        BankAccount.count += 1


a1 = BankAccount("Alice", 1000)
a2 = BankAccount("Bob",    500)
a3 = BankAccount("Carol",  250)

print(a1.bank_name)                        # PyBank
print(BankAccount.bank_name)               # PyBank — also via class name
print(f"Total accounts opened: {BankAccount.count}")  # 3

print(a1.owner, a2.owner, a3.owner)        # different per instance

PyBank
PyBank
Total accounts opened: 3
Alice Bob Carol


## 6. Where OOP Comes Into Action for ML

Everything up to here was setup. Now we'll see where OOP actually pays off.

**PyTorch** — the most popular deep learning framework, which you'll learn about in depth later this semester — is heavily *pythonic*. It's built entirely around the OOP concepts from this notebook. You don't need to understand what these models *do* yet; the point right now is to **recognize the patterns**. Every single idea below already appeared above.

### 6.1 A neural network is just a class

Here's a standard PyTorch model. Read it as an OOP exercise, not an ML one, and spot the pieces you already know:

- It **inherits** from `nn.Module` (the parent class) — that's `class NeuralNetwork(nn.Module)`.
- Its constructor calls **`super().__init__()`** — the exact `super()` line we flagged in the Inheritance section.
- It stores layers as **instance attributes** via `self.` in `__init__`.
- It **overrides** a method, `forward()`, to define its own behavior — polymorphism, just like `describe()`.

In [80]:
import torch
from torch import nn

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()                       # inherit from nn.Module
        self.flatten = nn.Flatten()              # layers stored as instance attributes
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):                        # override — defines this model's behavior
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits


model = NeuralNetwork()
print(model)                 # nn.Module defines __str__ for you — readable output

# You never call model.forward(x) directly. You call model(x).
# nn.Module implements __call__, which routes to forward() — the exact dunder pattern
# from the reference table above.

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Notice the last comment: you write `model(x)`, not `model.forward(x)`. That works because `nn.Module` defines `__call__`, which internally calls your `forward()`. That's the `__call__` dunder from Section 4, doing real work in a real framework.

### 6.2 A dataset is just a class too

When you load data in PyTorch, you write a class that inherits from `Dataset` and implements two dunder methods you already met: **`__len__`** and **`__getitem__`**.

- `__len__` tells PyTorch how many samples exist — same dunder as `len(vector)`.
- `__getitem__` lets PyTorch grab sample `i` with `dataset[i]` — the index-access dunder from the reference table.

That's the entire contract. PyTorch's `DataLoader` then uses these two methods to batch and shuffle your data automatically, without knowing anything about your specific dataset (polymorphism again).

In [ ]:
import os
import pandas as pd
from torch.utils.data import Dataset
from torchvision.io import decode_image

class CustomImageDataset(Dataset):
    def __init__(self, annotations_file, img_dir, transform=None, target_transform=None):
        self.img_labels = pd.read_csv(annotations_file)   # instance attributes
        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):                                    # len(dataset) -> number of samples
        return len(self.img_labels)

    def __getitem__(self, idx):                           # dataset[idx] -> one (image, label) pair
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        image = decode_image(img_path)
        label = self.img_labels.iloc[idx, 1]
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        return image, label

# (This cell won't run without an actual annotations file + image directory —
#  it's here to show the OOP structure, not to execute.)

Look back at the two classes above and match each line to a concept from this notebook:

| PyTorch code | OOP concept | Where you saw it |
|---|---|---|
| `class NeuralNetwork(nn.Module)` | Inheritance | Section 3.3 |
| `super().__init__()` | Calling the parent constructor | Section 3.3 |
| `self.flatten = ...` | Instance attributes | Sections 2, 5 |
| `def forward(self, x)` | Method override / polymorphism | Sections 3.3, 3.4 |
| `model(x)` calls `forward` | `__call__` dunder | Section 4 |
| `def __len__(self)` | `__len__` dunder | Section 4.2 |
| `def __getitem__(self, idx)` | `__getitem__` dunder | Section 4 reference table |

None of this is new. PyTorch didn't invent special syntax, it just leaned all the way into Python's OOP. That's why this recitation exists before you touch the framework.

## Wrap-Up

| Concept | Key idea |
|---|---|
| **Class / Object** | Blueprint and concrete instance |
| **`__init__` / `self`** | Constructor; reference to this specific object |
| **Encapsulation** | Bundle data + methods; control access with `_` / `__` |
| **Abstraction** | Hide internal complexity; expose a clean public interface |
| **Inheritance** | Child class reuses and extends parent via `super()` |
| **Polymorphism** | Same method name, different behavior per class |
| **Dunder methods** | Control how objects behave with `print`, `len`, `+`, `==`, `<`, `[]`, `()` |
| **Class attributes** | Shared across all instances (vs per-object instance attributes) |
| **PyTorch models & datasets** | Real classes built on every concept above |

**Next steps:**
- Run through the companion **Quick Check** notebook to self-test on these concepts.
- Rewrite the `Vector` class from a blank cell without looking — if you can, the dunder methods have landed.
- When PyTorch shows up later in the course, come back to Section 6. Every model and dataset you write will follow these exact patterns.

**Further reading:**
- [Python OOP — GeeksForGeeks](https://www.geeksforgeeks.org/interview-prep/object-oriented-programming-oop-tutorial/)
- [Abstract classes in Python](https://www.geeksforgeeks.org/python/abstract-classes-in-python/)
- [Python dunder methods](https://www.geeksforgeeks.org/dunder-magic-methods-python/)
- [PyTorch: Build the Neural Network](https://pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html)
- [PyTorch: Datasets & DataLoaders](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html)